In [0]:
from pyspark.sql.types import *
schema = StructType([
  StructField('customer_id', LongType()),
  StructField('created_timestamp', TimestampType()),
  StructField('customer_name', StringType()),
  StructField('date_of_birth', DateType()),
  StructField('email', StringType()),
  StructField('telephone', StringType()),
  StructField('member_since', DateType())
])

df_customers = spark.readStream.schema(schema).format('json').load('/Volumes/gizmobox/landing/streaming_data/')


In [0]:
display(df_customers)

In [0]:
customers_query = df_customers.writeStream.option(
    "checkpointLocation",
    "/Volumes/gizmobox/bronze/checkpoints/customers"
).trigger(
    availableNow=True
).toTable("gizmobox.bronze.customers_stream")

In [0]:
%sql
select * from gizmobox.bronze.customers_stream

![image_1773176856672.png](./image_1773176856672.png "image_1773176856672.png")

![image_1773176970663.png](./image_1773176970663.png "image_1773176970663.png")

Spark Structured Streaming, the combination of a write-ahead log (or WAL) and checkpointing is what ensures fault tolerance. Here’s how it works:

First, the write-ahead log stores all the incoming records before they’re processed. So, every event is written to this log as soon as it arrives, before any transformations or actions happen. Then, checkpointing keeps track of the progress of the stream—meaning it records exactly which data has been processed so far, what offsets or batch IDs were handled, and the exact state of the processing.

If a failure occurs—like a machine crash—Spark can recover by reading the write-ahead log to re-ingest any lost events, and it uses the checkpoint to restore the progress—so Spark knows exactly which data was successfully processed and which needs reprocessing. This way, you avoid duplicates, and you never lose any records. It’s really a combination of these two mechanisms—WAL ensures all input is safely logged, and checkpointing ensures that progress is remembered—so the whole pipeline can resume smoothly from where it left off.